# ExploitGen — Train trên Google Colab (NT521)

Notebook này train lại ExploitGen bằng `codebert-base` (thay `pseudo/fg-codebert` không công khai).

**TRƯỚC KHI CHẠY:** menu **Runtime → Change runtime type → T4 GPU**, rồi **Runtime → Run all**.

Thứ tự: kiểm tra GPU → lấy code → cài thư viện → tải model → (gắn Drive) → train Assembly → train Python.
Checkpoint tốt nhất lưu ở Google Drive: `MyDrive/NT521_ExploitGen/<lang>/checkpoint-best-rouge/`.

## 1. Kiểm tra GPU (phải thấy Tesla T4)

In [ ]:
!nvidia-smi

## 2. Lấy mã nguồn
Sửa `REPO_URL` thành repo GitHub của nhóm (chứa thư mục `ExploitGen/`).
Nếu KHÔNG dùng GitHub: bỏ qua cell này, bấm biểu tượng thư mục bên trái để upload `ExploitGen.zip` rồi `!unzip ExploitGen.zip`.

In [ ]:
import os
REPO_URL = "https://github.com/HLong145/NT521_project.git"  #@param {type:"string"}
SUBDIR   = "ExploitGen"  # thư mục chứa model.py, train_exploitgen.py, data/

repo_name = REPO_URL.rstrip('/').split('/')[-1].replace('.git','')
if not os.path.isdir(repo_name):
    !git clone $REPO_URL
%cd $repo_name/$SUBDIR
print('CWD:', os.getcwd())
assert os.path.exists('train_exploitgen.py'), 'Khong thay train_exploitgen.py - kiem tra REPO_URL/SUBDIR'
assert os.path.exists('data/python/train.csv'), 'Khong thay data/python/train.csv'

## 3. Cài thư viện
Giữ nguyên `torch` (có CUDA) của Colab, chỉ cài đúng phiên bản ExploitGen cần.

In [ ]:
!pip -q install "transformers==4.21.0" "huggingface_hub==0.25.2" "tokenizers==0.12.1" \
                "easy-rouge>=0.2.2"
import torch; print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())

## 4. Tải codebert-base về thư mục local
Tải sẵn rồi `from_pretrained(local_dir)` để né bug tải của transformers 4.21.

In [ ]:
import os
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
from huggingface_hub import snapshot_download
MODEL_DIR = snapshot_download('microsoft/codebert-base', local_dir='codebert-base',
                             local_dir_use_symlinks=False)
print('MODEL DIR:', MODEL_DIR)
print(sorted(os.listdir(MODEL_DIR)))

## 5. Gắn Google Drive (để lưu checkpoint, tránh mất khi hết phiên)
Bỏ qua cell này nếu không muốn dùng Drive — khi đó đổi `SAVE_ROOT` ở cell dưới thành `runs`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE_ROOT = '/content/drive/MyDrive/NT521_ExploitGen'
import os; os.makedirs(SAVE_ROOT, exist_ok=True)
print('Luu checkpoint vao:', SAVE_ROOT)

## 6. TRAIN ASSEMBLY (nhẹ ~3k mẫu — chạy trước để kiểm tra GPU, ~1–1.5h)
Nếu gặp **CUDA OOM**: đổi `--batch_size 16` → `8` và thêm `--grad_accum 2`.

In [ ]:
!python train_exploitgen.py --lang assembly --model_path codebert-base \
    --output_dir "$SAVE_ROOT/assembly" \
    --epochs 50 --batch_size 16 --eval_batch_size 32

## 7. TRAIN PYTHON (nặng ~14.8k mẫu, ~3–5h; code tự early-stop)

In [ ]:
!python train_exploitgen.py --lang python --model_path codebert-base \
    --output_dir "$SAVE_ROOT/python" \
    --epochs 50 --batch_size 16 --eval_batch_size 32

## 8. Kết quả
- Checkpoint tốt nhất: `$SAVE_ROOT/<lang>/checkpoint-best-rouge/pytorch_model.bin`
- Kết quả test (ROUGE-W, Exact Match) in ở cuối log mỗi lần train và lưu ở `<output_dir>/test_output/`.
- Tải 2 file `pytorch_model.bin` (assembly + python) về máy để demo local (xem `HUONG_DAN_TRAIN.md` Mục 5).